# Table Linearization (Stufe 1.5)

Nimmt die strukturierten JSONs aus `structured_export.ipynb` und linearisiert Tabellen per lokalem LLM (LM Studio).

| Vorher | Nachher |
|---|---|
| `tables[].html` — HTML-Tabelle | `tables[].linearized` — Key-Value-Text |
| Spalten/Zeilen nur via Grid-Struktur zuordenbar | Jeder Fakt ist eigenständig und suchbar |

## LLM-Input
- **Format**: HTML (preserviert colspan/rowspan, kürzer als Markdown)
- **Kontext**: Section-Header + beschreibender Paragraph + doc_id/company_name

## Output-Format
```
Table: "Revenue Breakdown (Page 42)"
- North America, 2012: $478M (38% of segment)
- Europe and Africa, 2012: $528M (42% of segment)
```

## Resumability
- Fortschritt wird nach jeder Tabelle gespeichert
- Unterbrochene Läufe können fortgesetzt werden

In [1]:
import json
import re
import time
import datetime
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from tqdm import tqdm
import requests

In [2]:
SYSTEM_PROMPT = (
    "You linearize HTML tables into bullet-point lists.\n"
    "\n"
    "RULES:\n"
    "1. Combine related values from the same row into ONE bullet\n"
    "   (e.g., $-amount and %-share belong together).\n"
    "2. Keep footnote markers (1), (2) next to the label, NOT inside\n"
    "   every data row.\n"
    "3. Place the full footnote text as a separate line AFTER all bullets.\n"
    "4. Copy all proper nouns, chemical names, and product names\n"
    "   CHARACTER-FOR-CHARACTER. Do not paraphrase them.\n"
    "\n"
    "EXAMPLE INPUT:\n"
    "<table>\n"
    "  <tr><td></td><th>2012</th><th>2011</th></tr>\n"
    "  <tr><th>Region A</th><td>100</td><td>90</td></tr>\n"
    "  <tr><th>Region B</th><td>50</td><td>45</td></tr>\n"
    "  <tr><th>Total(1)</th><td>150</td><td>135</td></tr>\n"
    "</table>\n"
    "Footnotes:\n"
    "  (1) Excludes inter-segment sales of $10 million and $8 million.\n"
    "\n"
    "EXAMPLE OUTPUT:\n"
    "Table: Example, Page X\n"
    "- Region A, 2012: $100 million\n"
    "- Region A, 2011: $90 million\n"
    "- Region B, 2012: $50 million\n"
    "- Region B, 2011: $45 million\n"
    "- Total(1), 2012: $150 million\n"
    "- Total(1), 2011: $135 million\n"
    "(1) Excludes inter-segment sales of $10 million and $8 million."
)

TABLE_CONTEXT_KEYWORDS = frozenset({
    "table", "following", "summarize", "set forth",
    "presents", "illustrate", "below", "shows",
})

FOOTNOTE_MARKER_RE = re.compile(r'^\(?(\d{1,2})\)?$|^\*+$|^\u2020$|^\u2021$')


class TableLinearizer:
    """Structured JSON tables -> linearized key-value text via local LLM."""

    def __init__(self, source_dir: str, output_dir: str = None,
                 endpoint: str = "http://localhost:1234/v1",
                 model_name: str = "", temperature: float = 0.1,
                 max_tokens: int = 2048, max_context: int = 8192):
        self.source_dir = Path(source_dir)
        if output_dir:
            self.output_dir = Path(output_dir)
        else:
            self.output_dir = self.source_dir.parent / (self.source_dir.name + "_linearized")
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.endpoint = endpoint.rstrip("/")
        if not self.endpoint.endswith("/v1"):
            self.endpoint += "/v1"
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.max_context = max_context

        print(f"Source: {self.source_dir}")
        print(f"Output: {self.output_dir}")
        print(f"LLM:    {self.endpoint} / {self.model_name}")

    # -- Token estimation -------------------------------------------------

    @staticmethod
    def _estimate_tokens(text: str) -> int:
        """Rough estimate: ~4 chars per token for English/HTML."""
        return len(text) // 4

    # -- LLM interaction --------------------------------------------------

    def _call_llm(self, user_prompt: str) -> Tuple[str, int, int, float, str]:
        start = time.time()
        resp = requests.post(
            f"{self.endpoint}/chat/completions",
            json={
                "model": self.model_name,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                "temperature": self.temperature,
                "max_tokens": self.max_tokens,
            },
            timeout=120,
        )
        resp.raise_for_status()
        result = resp.json()
        elapsed = time.time() - start

        if "error" in result:
            raise RuntimeError(f"LLM API error: {result['error']}")
        if "choices" not in result:
            raise RuntimeError(f"Unexpected API response (no 'choices'): {json.dumps(result)[:500]}")

        text = result["choices"][0]["message"]["content"]
        usage = result.get("usage", {})
        finish_reason = result["choices"][0].get("finish_reason", "unknown")
        return (
            text,
            usage.get("prompt_tokens", 0),
            usage.get("completion_tokens", 0),
            elapsed,
            finish_reason,
        )

    # -- Preflight --------------------------------------------------------

    def _preflight_check(self):
        """Test LLM connectivity before processing."""
        try:
            text, _, _, elapsed, _ = self._call_llm("Say 'ok'.")
            print(f"Preflight OK ({elapsed:.1f}s): {text[:80]}")
        except Exception as e:
            raise RuntimeError(
                f"Preflight failed — is the model loaded?\n{e}"
            ) from e

    # -- Context extraction -----------------------------------------------

    def _build_table_context_map(self, data: Dict) -> Dict[int, Dict]:
        """Map table_id -> context dict by scanning content pages."""
        metainfo = data["metainfo"]
        context_map: Dict[int, Dict] = {}
        last_section_header: Optional[str] = None

        for page in data.get("content", []):
            page_num = page["page"]
            blocks = page.get("content", [])

            for i, block in enumerate(blocks):
                btype = block.get("type", "")

                if btype == "section_header":
                    last_section_header = block.get("text", "")

                if btype != "table":
                    continue

                tid = block["table_id"]
                section = None
                description = None

                # Backward scan for section header and description
                for j in range(i - 1, -1, -1):
                    b = blocks[j]
                    bt = b.get("type", "")
                    text = b.get("text", "")
                    if bt == "section_header":
                        if section is None:
                            section = text
                        break
                    elif bt == "paragraph" and description is None:
                        text_lower = text.lower()
                        if any(kw in text_lower for kw in TABLE_CONTEXT_KEYWORDS):
                            description = text

                # Forward scan for footnotes
                footnotes = []
                for k in range(i + 1, len(blocks)):
                    b = blocks[k]
                    bt = b.get("type", "")
                    txt = b.get("text", "").strip()
                    if bt in ("page_footer", "picture"):
                        continue
                    if bt == "paragraph" and re.match(r'^[_\-]{5,}$', txt):
                        continue
                    if bt == "footnote":
                        footnotes.append(txt)
                        continue
                    if bt == "paragraph" and re.match(r'^\(?\d+\)', txt):
                        footnotes.append(txt)
                        continue
                    break

                context_map[tid] = {
                    "section": section or last_section_header or "",
                    "description": description,
                    "footnotes": footnotes,
                    "doc_id": metainfo.get("doc_id", ""),
                    "company_name": metainfo.get("company_name", ""),
                    "page": page_num,
                }

        return context_map

    # -- Prompt building --------------------------------------------------

    def _build_user_prompt(self, html: str, context: Dict) -> str:
        parts = ["NOW LINEARIZE THE FOLLOWING TABLE:", "",
                 f"Document: {context['doc_id']} ({context['company_name']})"]
        if context.get("section"):
            parts.append(f"Section: {context['section']}")
        if context.get("description"):
            parts.append(f"Description: {context['description']}")
        parts.append(f"Page: {context['page']}")
        parts.append("")
        parts.append(html)
        if context.get("footnotes"):
            parts.append("")
            parts.append("Footnotes:")
            for fn in context["footnotes"]:
                parts.append(f"  {fn}")
        return "\n".join(parts)

    # -- Validation -------------------------------------------------------

    @staticmethod
    def _validate_output(text: str) -> bool:
        """Check that output starts with Table: and has at least one '- ' line."""
        lines = text.strip().split("\n")
        has_table = any(line.strip().startswith("Table:") for line in lines)
        has_data = any(line.strip().startswith("- ") for line in lines)
        return has_table and has_data

    # -- Table grid cleanup -----------------------------------------------

    @staticmethod
    def _clean_table_grid(grid: Dict) -> Dict:
        """Remove footnote-marker columns and merge markers into preceding cells."""
        cells = grid.get("cells", [])
        num_cols = grid.get("num_cols", 0)
        num_rows = grid.get("num_rows", 0)
        if not cells or num_cols < 2:
            return grid

        # Group cells by column
        col_cells: Dict[int, List[Dict]] = {}
        for c in cells:
            col_cells.setdefault(c["col"], []).append(c)

        # Identify marker columns: all headers empty, all data match pattern
        marker_cols = set()
        for col_idx in range(num_cols):
            cc = col_cells.get(col_idx, [])
            if not cc:
                continue
            if any(c.get("col_span", 1) > 1 for c in cc):
                continue
            headers = [c for c in cc if c.get("is_col_header")]
            data = [c for c in cc if not c.get("is_col_header")]
            if any(c["text"].strip() for c in headers):
                continue
            if not data:
                continue
            if not all(
                not c["text"].strip() or FOOTNOTE_MARKER_RE.match(c["text"].strip())
                for c in data
            ):
                continue
            if any(c["text"].strip() and FOOTNOTE_MARKER_RE.match(c["text"].strip()) for c in data):
                marker_cols.add(col_idx)

        if not marker_cols:
            return grid

        # Shallow-copy cells to avoid mutating originals
        cells = [{**c} for c in cells]
        cell_lookup = {(c["row"], c["col"]): c for c in cells}

        # Merge markers into preceding cell
        new_cells = []
        for c in cells:
            if c["col"] in marker_cols:
                txt = c["text"].strip()
                if txt:
                    prev_col = c["col"] - 1
                    while prev_col >= 0 and prev_col in marker_cols:
                        prev_col -= 1
                    prev = cell_lookup.get((c["row"], prev_col))
                    if prev and prev["text"].strip():
                        prev["text"] = prev["text"].rstrip() + " " + txt
                continue
            new_cells.append(c)

        # Remap col indices and adjust colspan
        kept_cols = sorted(c for c in range(num_cols) if c not in marker_cols)
        col_remap = {old: new for new, old in enumerate(kept_cols)}
        for c in new_cells:
            orig_col = c["col"]
            c["col"] = col_remap[orig_col]
            cs = c.get("col_span", 1)
            if cs > 1:
                spanned = range(orig_col, orig_col + cs)
                c["col_span"] = sum(1 for s in spanned if s not in marker_cols)

        return {
            "num_rows": num_rows,
            "num_cols": num_cols - len(marker_cols),
            "cells": new_cells,
        }

    @staticmethod
    def _grid_to_html(grid: Dict) -> str:
        """Render a cell grid as HTML table."""
        cells = grid.get("cells", [])
        num_rows = grid.get("num_rows", 0)
        num_cols = grid.get("num_cols", 0)
        if not cells or num_rows == 0:
            return ""

        lookup = {(c["row"], c["col"]): c for c in cells}

        # Track cells covered by rowspan/colspan
        covered = set()
        for c in cells:
            for dr in range(c.get("row_span", 1)):
                for dc in range(c.get("col_span", 1)):
                    if dr == 0 and dc == 0:
                        continue
                    covered.add((c["row"] + dr, c["col"] + dc))

        lines = ["<table>"]
        for r in range(num_rows):
            lines.append("  <tr>")
            for col in range(num_cols):
                if (r, col) in covered:
                    continue
                c = lookup.get((r, col))
                if c is None:
                    lines.append("    <td></td>")
                    continue
                tag = "th" if c.get("is_col_header") or c.get("is_row_header") else "td"
                attrs = ""
                rs = c.get("row_span", 1)
                cs = c.get("col_span", 1)
                if rs > 1:
                    attrs += f' rowspan="{rs}"'
                if cs > 1:
                    attrs += f' colspan="{cs}"'
                lines.append(f"    <{tag}{attrs}>{c['text']}</{tag}>")
            lines.append("  </tr>")
        lines.append("</table>")
        return "\n".join(lines)

    # -- Single table processing ------------------------------------------

    def _linearize_table(self, table: Dict, context: Dict) -> Dict:
        tid = table["table_id"]
        page = table.get("page", context.get("page", 0))

        # Build HTML: prefer cleaned grid, fall back to raw HTML
        grid = table.get("json", {})
        if grid.get("cells"):
            cleaned = self._clean_table_grid(grid)
            html = self._grid_to_html(cleaned)
        else:
            html = table.get("html", "")

        if not html.strip():
            return {"table_id": tid, "page": page,
                    "linearized": "", "status": "skipped"}

        prompt = self._build_user_prompt(html, context)
        est_tokens = (self._estimate_tokens(SYSTEM_PROMPT)
                      + self._estimate_tokens(prompt))

        if est_tokens > self.max_context - self.max_tokens:
            return {"table_id": tid, "page": page,
                    "linearized": "", "status": "error",
                    "prompt": prompt,
                    "error": f"too_large ({est_tokens} est. tokens)"}

        try:
            text, tok_in, tok_out, elapsed, finish_reason = self._call_llm(prompt)

            if not self._validate_output(text):
                return {
                    "table_id": tid, "page": page,
                    "linearized": "", "status": "error_validation",
                    "prompt": prompt,
                    "output_raw": text,
                    "model": self.model_name,
                    "tokens_in": tok_in, "tokens_out": tok_out,
                    "elapsed_s": round(elapsed, 2),
                }

            truncated = (finish_reason == "length")
            return {
                "table_id": tid, "page": page,
                "linearized": text.strip(),
                "status": "ok_truncated" if truncated else "ok",
                "prompt": prompt,
                "model": self.model_name,
                "tokens_in": tok_in, "tokens_out": tok_out,
                "elapsed_s": round(elapsed, 2),
                "finish_reason": finish_reason,
            }
        except Exception as e:
            return {"table_id": tid, "page": page,
                    "linearized": "", "status": "error",
                    "prompt": prompt,
                    "error": str(e)}

    # -- Resumability -----------------------------------------------------

    def _load_progress(self, doc_id: str) -> Optional[Dict]:
        out_file = self.output_dir / f"{doc_id}.json"
        if out_file.exists():
            with open(out_file, "r", encoding="utf-8") as f:
                return json.load(f)
        return None

    def _save_progress(self, doc_id: str, metainfo: Dict,
                       table_results: List[Dict]):
        out_file = self.output_dir / f"{doc_id}.json"
        ok = sum(1 for t in table_results if t.get("status") == "ok")
        truncated = sum(1 for t in table_results
                        if t.get("status") == "ok_truncated")
        err = sum(1 for t in table_results
                  if t.get("status", "").startswith("error"))
        skipped = sum(1 for t in table_results if t.get("status") == "skipped")

        output = {
            "metainfo": metainfo,
            "tables": table_results,
            "linearization_info": {
                "model": self.model_name,
                "endpoint": self.endpoint,
                "completed_at": datetime.datetime.now().isoformat(),
                "tables_total": len(table_results),
                "tables_ok": ok,
                "tables_truncated": truncated,
                "tables_error": err,
                "tables_skipped": skipped,
            },
        }
        with open(out_file, "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

    # -- Document processing ----------------------------------------------

    def _process_document(self, data: Dict) -> List[Dict]:
        doc_id = data["metainfo"]["doc_id"]
        tables = data.get("tables", [])

        if not tables:
            return []

        # Check for existing progress (resumability)
        existing = self._load_progress(doc_id)
        done_ids: set = set()
        results: List[Dict] = []
        if existing:
            for t in existing.get("tables", []):
                if t.get("status") in ("ok", "skipped"):
                    done_ids.add(t["table_id"])
                    results.append(t)

        context_map = self._build_table_context_map(data)

        remaining = [t for t in tables if t["table_id"] not in done_ids]
        if not remaining:
            tqdm.write(f"  {doc_id}: all {len(results)} tables already done")
            return results

        if done_ids:
            tqdm.write(f"  {doc_id}: resuming, {len(done_ids)} done, "
                       f"{len(remaining)} remaining")

        for table in tqdm(remaining, desc=f"  {doc_id}", leave=False):
            context = context_map.get(table["table_id"], {
                "section": "", "description": None,
                "footnotes": [],
                "doc_id": doc_id,
                "company_name": data["metainfo"].get("company_name", ""),
                "page": table.get("page", 0),
            })

            result = self._linearize_table(table, context)
            results.append(result)

            # Save after each table for resumability
            self._save_progress(doc_id, data["metainfo"], results)

        return results

    # -- Batch processing -------------------------------------------------

    def process_all(self) -> Dict:
        json_files = sorted(self.source_dir.glob("*.json"))
        print(f"{len(json_files)} JSON files\n")

        self._preflight_check()

        stats = {"success": 0, "error": 0,
                 "tables_total": 0, "tables_ok": 0, "tables_truncated": 0}

        for jf in tqdm(json_files, desc="Linearizing"):
            try:
                with open(jf, "r", encoding="utf-8") as f:
                    data = json.load(f)

                results = self._process_document(data)

                ok = sum(1 for t in results if t.get("status") == "ok")
                trunc = sum(1 for t in results
                            if t.get("status") == "ok_truncated")
                err = sum(1 for t in results
                          if t.get("status", "").startswith("error"))
                skipped = sum(1 for t in results
                              if t.get("status") == "skipped")

                tqdm.write(f"  {jf.stem}: {len(results)} tables "
                           f"(ok={ok}, trunc={trunc}, err={err}, skipped={skipped})")
                stats["success"] += 1
                stats["tables_total"] += len(results)
                stats["tables_ok"] += ok
                stats["tables_truncated"] += trunc

            except Exception as e:
                tqdm.write(f"  {jf.stem}: Error — {e}")
                stats["error"] += 1

        print(f"\nDocs: {stats['success']} OK, {stats['error']} errors")
        print(f"Tables: {stats['tables_ok']}/{stats['tables_total']} linearized"
              f" ({stats['tables_truncated']} truncated)")
        return stats

In [3]:
linearizer = TableLinearizer(
    source_dir="C:/Users/phili/PycharmProjects/doc/data/src/docling_output/20260409_195023",
    endpoint="http://127.0.0.1:1234",
    model_name="google/gemma-4-e4b",  # <- Name des in LM Studio geladenen Modells
    temperature=0.0,
    max_tokens=12000,
    max_context=131072,
)
linearizer.process_all()

Source: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195023
Output: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195023_linearized
LLM:    http://127.0.0.1:1234/v1 / google/gemma-4-e4b
6 JSON files

Preflight OK (1.6s): ok


Linearizing:  17%|█▋        | 1/6 [25:48<2:09:03, 1548.62s/it] 

  CMCSA_2015: 121 tables (ok=92, trunc=0, err=29, skipped=0)



Linearizing:  33%|███▎      | 2/6 [47:05<1:32:36, 1389.02s/it][A

  CME_2010: 104 tables (ok=85, trunc=0, err=19, skipped=0)


...


Linearizing:  83%|████████▎ | 5/6 [1:52:42<23:17, 1397.90s/it][A

  DRE_2016: 112 tables (ok=90, trunc=0, err=22, skipped=0)



Linearizing: 100%|██████████| 6/6 [2:21:22<00:00, 1413.77s/it][A

  DVN_2010: 135 tables (ok=109, trunc=0, err=26, skipped=0)

Docs: 6 OK, 0 errors
Tables: 540/667 linearized (0 truncated)


{'success': 6,
 'error': 0,
 'tables_total': 667,
 'tables_ok': 540,
 'tables_truncated': 0}